# 📈 Stock Price Prediction — LSTM vs Random Forest
**Course Project | AI & Machine Learning**

---

## Project Overview

This notebook implements an end-to-end stock price prediction pipeline that:
1. Downloads historical price data for **AAPL, GOOG, MSFT, AMZN** via `yfinance`
2. Performs **Exploratory Data Analysis** — closing prices, volume, moving averages, daily returns, and correlation
3. Trains an **LSTM (Long Short-Term Memory)** deep learning model on Apple's historical prices
4. Trains a **Random Forest** baseline model on the same data
5. Compares both models side-by-side using **RMSE** and a shared prediction chart
6. Generates a **60-day future price forecast**

---

### Architecture
The code is structured using **Object-Oriented Programming** with an Abstract Base Class (`BaseModel`) that enforces a consistent interface across all model implementations.

```
config.py          → All hyperparameters in one place
data_pipeline.py   → Download, scale, sequence creation, train/test split
visualizer.py      → All plots
models/
  base_model.py    → Abstract Base Class (ABC)
  lstm_model.py    → LSTM implementation
  rf_model.py      → Random Forest implementation
main.py            → Orchestrator
```


## ⚙️ Section 1 — Install Dependencies

In [ ]:
# Run this cell once if packages are missing
# !pip install yfinance pandas-datareader scikit-learn tensorflow keras matplotlib seaborn
print("All packages ready.")

## 🔧 Section 2 — Configuration (`config.py`)

All tunable hyperparameters live in one place so experiments are a one-liner change.


In [ ]:
from datetime import datetime

# ── Tickers ───────────────────────────────────────────────────────────────────
TECH_LIST    = ["AAPL", "GOOG", "MSFT", "AMZN"]
COMPANY_NAMES = ["APPLE", "GOOGLE", "MICROSOFT", "AMAZON"]

PRIMARY_TICKER      = "AAPL"
PRIMARY_TICKER_NAME = "Apple"

# ── Date Ranges ───────────────────────────────────────────────────────────────
EDA_END_DATE   = datetime.now()
EDA_START_DATE = datetime(EDA_END_DATE.year - 1, EDA_END_DATE.month, EDA_END_DATE.day)

LSTM_START_DATE = "2012-01-01"
LSTM_END_DATE   = datetime.now()

# ── Data Pipeline ─────────────────────────────────────────────────────────────
LOOKBACK_WINDOW    = 60       # past days used as input features
TRAIN_SPLIT_RATIO  = 0.95     # 95 % train, 5 % test
MOVING_AVG_WINDOWS = [10, 20, 50]

# ── LSTM Hyperparameters ──────────────────────────────────────────────────────
LSTM_UNITS_LAYER_1 = 128
LSTM_UNITS_LAYER_2 = 64
DENSE_UNITS        = 25
EPOCHS             = 1
BATCH_SIZE         = 1
OPTIMIZER          = "adam"
LOSS               = "mean_squared_error"

# ── Random Forest Hyperparameters ─────────────────────────────────────────────
RF_N_ESTIMATORS = 100
RF_RANDOM_STATE = 42

# ── Forecasting ───────────────────────────────────────────────────────────────
NUM_FUTURE_DAYS = 60

print("✅ Config loaded.")

## 📦 Section 3 — Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
from abc import ABC, abstractmethod

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor

from keras.models import Sequential
from keras.layers import Dense, LSTM

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.style.use("fivethirtyeight")
%matplotlib inline

print("✅ Imports complete.")

## 📥 Section 4 — Data Download & EDA (`data_pipeline.py`)

### 4.1 Download 1-Year EDA Data for All Tickers


In [ ]:
def download_eda_data():
    """Download 1-year price data for every ticker and compute MA + Daily Return."""
    company_list = []
    for ticker, name in zip(TECH_LIST, COMPANY_NAMES):
        df = yf.download(ticker, EDA_START_DATE, EDA_END_DATE, progress=False)
        df["company_name"] = name
        for window in MOVING_AVG_WINDOWS:
            df[f"MA for {window} days"] = df["Close"].rolling(window).mean()
        df["Daily Return"] = df["Close"].pct_change()
        company_list.append(df)

    raw = yf.download(TECH_LIST, start=EDA_START_DATE, end=EDA_END_DATE, progress=False)
    closing_df = raw["Close"]
    return company_list, closing_df

company_list, closing_df = download_eda_data()
print(f"✅ Downloaded data for: {TECH_LIST}")
print(f"Sample — AAPL tail:")
company_list[0].tail(3)

## 📊 Section 5 — Exploratory Data Analysis (`visualizer.py`)

### 5.1 Closing Prices


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, company, ticker in zip(axes.flatten(), company_list, TECH_LIST):
    company["Close"].plot(ax=ax)
    ax.set_ylabel("Close (USD)")
    ax.set_xlabel(None)
    ax.set_title(f"Closing Price — {ticker}")
plt.suptitle("Daily Closing Prices (Last 1 Year)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### 5.2 Trading Volume

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, company, ticker in zip(axes.flatten(), company_list, TECH_LIST):
    company["Volume"].plot(ax=ax)
    ax.set_ylabel("Volume")
    ax.set_xlabel(None)
    ax.set_title(f"Sales Volume — {ticker}")
plt.suptitle("Daily Trading Volume (Last 1 Year)", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### 5.3 Moving Averages (10 / 20 / 50 Days)

In [ ]:
AAPL, GOOG, MSFT, AMZN = company_list
ma_cols = ["Close"] + [f"MA for {w} days" for w in MOVING_AVG_WINDOWS]

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, company, name in zip(axes.flatten(), company_list, COMPANY_NAMES):
    company[ma_cols].plot(ax=ax)
    ax.set_title(name)
plt.suptitle("Closing Price with Moving Averages", fontsize=16, y=1.02)
fig.tight_layout()
plt.show()

### 5.4 Daily Returns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, company, name in zip(axes.flatten(), company_list, COMPANY_NAMES):
    company["Daily Return"].plot(ax=ax, legend=True, linestyle="--", marker="o", markersize=3)
    ax.set_title(name)
plt.suptitle("Daily Return (%)", fontsize=16, y=1.02)
fig.tight_layout()
plt.show()

### 5.5 Return Correlation — Pair Plot

In [ ]:
tech_rets = closing_df.pct_change()
sns.pairplot(tech_rets.dropna(), kind="reg")
plt.suptitle("Pairwise Correlation of Daily Returns", y=1.02)
plt.show()

### 5.6 Correlation Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(tech_rets.corr(), annot=True, cmap="summer", ax=axes[0])
axes[0].set_title("Correlation — Daily Returns")

sns.heatmap(closing_df.corr(), annot=True, cmap="summer", ax=axes[1])
axes[1].set_title("Correlation — Closing Prices")

plt.tight_layout()
plt.show()

## 🔁 Section 6 — LSTM Data Preprocessing (`data_pipeline.py`)

### 6.1 Download Long-History Apple Data


In [ ]:
df = yf.download(PRIMARY_TICKER, start=LSTM_START_DATE, end=LSTM_END_DATE, progress=False)
print(f"Shape: {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.tail()

### 6.2 Close Price History

In [ ]:
plt.figure(figsize=(16, 6))
plt.title(f"{PRIMARY_TICKER_NAME} — Full Closing Price History", fontsize=16)
plt.plot(df["Close"], color="steelblue")
plt.xlabel("Date", fontsize=14)
plt.ylabel("Close Price USD ($)", fontsize=14)
plt.tight_layout()
plt.show()

### 6.3 Scale & Build 60-Day Lookback Sequences

The model takes the past **60 trading days** as input and predicts the next day's price.

- Input shape: `(samples, 60, 1)` — required by LSTM *and* handled internally by RF
- 95 % of data → training, 5 % → testing


In [ ]:
def build_lstm_datasets(df):
    data = df[["Close"]]
    dataset = data.values
    training_data_len = int(np.ceil(len(dataset) * TRAIN_SPLIT_RATIO))

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(dataset)

    # Training sequences
    train_data = scaled_data[:training_data_len, :]
    x_train, y_train = [], []
    for i in range(LOOKBACK_WINDOW, len(train_data)):
        x_train.append(train_data[i - LOOKBACK_WINDOW:i, 0])
        y_train.append(train_data[i, 0])

    x_train = np.array(x_train).reshape(-1, LOOKBACK_WINDOW, 1)
    y_train = np.array(y_train)

    # Test sequences
    test_data = scaled_data[training_data_len - LOOKBACK_WINDOW:, :]
    x_test, y_test = [], dataset[training_data_len:, :]
    for i in range(LOOKBACK_WINDOW, len(test_data)):
        x_test.append(test_data[i - LOOKBACK_WINDOW:i, 0])

    x_test = np.array(x_test).reshape(-1, LOOKBACK_WINDOW, 1)

    return x_train, y_train, x_test, y_test, scaler, training_data_len, scaled_data

x_train, y_train, x_test, y_test, scaler, training_data_len, scaled_data = build_lstm_datasets(df)

print(f"x_train shape : {x_train.shape}  → (samples, timesteps, features)")
print(f"y_train shape : {y_train.shape}")
print(f"x_test  shape : {x_test.shape}")
print(f"y_test  shape : {y_test.shape}")
print(f"Training boundary index: {training_data_len}")

## 🧠 Section 7 — Model Definitions

### 7.1 Abstract Base Class (`models/base_model.py`)

Using Python's `ABC` module to enforce a consistent interface.  
Any future model **must** implement `train()` and `predict()`.


In [ ]:
class BaseModel(ABC):
    """
    Common interface for all stock-prediction models.
    Subclasses must implement train() and predict().
    """

    @abstractmethod
    def train(self, x_train: np.ndarray, y_train: np.ndarray, **kwargs) -> None:
        ...

    @abstractmethod
    def predict(self, x: np.ndarray, **kwargs) -> np.ndarray:
        ...

    def evaluate(self, predictions: np.ndarray, actuals: np.ndarray) -> dict:
        """Default metric: RMSE. Override to add MAE, R², etc."""
        rmse = float(np.sqrt(np.mean((predictions - actuals) ** 2)))
        return {"rmse": rmse}

print("✅ BaseModel ABC defined.")

### 7.2 LSTM Model (`models/lstm_model.py`)

**Architecture:**

| Layer | Units | Notes |
|---|---|---|
| LSTM | 128 | `return_sequences=True` |
| LSTM | 64 | `return_sequences=False` |
| Dense | 25 | Fully connected |
| Dense | 1 | Output — next-day price |


In [ ]:
class LSTMModel(BaseModel):
    """Two-layer LSTM for univariate time-series forecasting."""

    def __init__(self):
        self._model = self._build()

    def _build(self) -> Sequential:
        model = Sequential([
            LSTM(LSTM_UNITS_LAYER_1, return_sequences=True,
                 input_shape=(LOOKBACK_WINDOW, 1)),
            LSTM(LSTM_UNITS_LAYER_2, return_sequences=False),
            Dense(DENSE_UNITS),
            Dense(1),
        ])
        model.compile(optimizer=OPTIMIZER, loss=LOSS)
        return model

    def train(self, x_train, y_train, **kwargs):
        epochs     = kwargs.pop("epochs",     EPOCHS)
        batch_size = kwargs.pop("batch_size", BATCH_SIZE)
        self._model.fit(x_train, y_train, epochs=epochs,
                        batch_size=batch_size, **kwargs)

    def predict(self, x, **kwargs):
        return self._model.predict(x, **kwargs)

    @property
    def keras_model(self):
        return self._model

print("✅ LSTMModel defined.")
# Print architecture summary
LSTMModel().keras_model.summary()

### 7.3 Random Forest Model (`models/rf_model.py`)

**Key design** — the RF flattens 3D pipeline data `(samples, 60, 1)` → `(samples, 60)` internally,  
so `main.py` calls both models with the exact same API.


In [ ]:
class RFModel(BaseModel):
    """
    Random Forest baseline wrapped in the BaseModel interface.
    Handles 3D → 2D reshaping internally so callers use the same API as LSTM.
    """

    def __init__(self, n_estimators=RF_N_ESTIMATORS, random_state=RF_RANDOM_STATE):
        self._model = RandomForestRegressor(
            n_estimators=n_estimators,
            random_state=random_state,
            n_jobs=-1,
        )

    @staticmethod
    def _flatten(x: np.ndarray) -> np.ndarray:
        """(samples, timesteps, 1) → (samples, timesteps)"""
        return x.reshape(x.shape[0], -1) if x.ndim == 3 else x

    def train(self, x_train, y_train, **kwargs):
        print(f"Training RandomForestRegressor (n_estimators={self._model.n_estimators})…")
        self._model.fit(self._flatten(x_train), y_train)
        print("✅ Random Forest training complete.")

    def predict(self, x, **kwargs) -> np.ndarray:
        """Returns shape (samples, 1) — same contract as LSTMModel.predict()"""
        return self._model.predict(self._flatten(x)).reshape(-1, 1)

    @property
    def feature_importances(self):
        return self._model.feature_importances_

print("✅ RFModel defined.")

## 🏋️ Section 8 — Training (`main.py`)

### 8.1 Train LSTM


In [ ]:
lstm = LSTMModel()
lstm.train(x_train, y_train)
print("✅ LSTM training complete.")

### 8.2 Train Random Forest

In [ ]:
rf = RFModel()
rf.train(x_train, y_train)

## 📐 Section 9 — Evaluation & Comparison

### 9.1 Generate Predictions


In [ ]:
# LSTM
scaled_preds_lstm = lstm.predict(x_test)
predictions_lstm  = scaler.inverse_transform(scaled_preds_lstm)

# Random Forest
scaled_preds_rf = rf.predict(x_test)
predictions_rf  = scaler.inverse_transform(scaled_preds_rf)

print(f"LSTM predictions shape : {predictions_lstm.shape}")
print(f"RF   predictions shape : {predictions_rf.shape}")

### 9.2 RMSE Comparison

In [ ]:
metrics_lstm = lstm.evaluate(predictions_lstm, y_test)
metrics_rf   = rf.evaluate(predictions_rf,   y_test)

print("─" * 40)
print(f"  {'Model':<22} {'RMSE':>10}")
print("─" * 40)
print(f"  {'LSTM':<22} {metrics_lstm['rmse']:>10.4f}")
print(f"  {'Random Forest':<22} {metrics_rf['rmse']:>10.4f}")
print("─" * 40)

better = "LSTM" if metrics_lstm['rmse'] < metrics_rf['rmse'] else "Random Forest"
print(f"\n  ✅ Lower RMSE → {better}")

### 9.3 Attach Predictions to Validation Slice

In [ ]:
data_close   = df[["Close"]]
train_slice  = data_close[:training_data_len]
valid_slice  = data_close[training_data_len:].copy()

valid_slice["Predictions"]    = predictions_lstm[:len(valid_slice)]
valid_slice["RF Predictions"] = predictions_rf[:len(valid_slice)]

valid_slice.tail()

## 📉 Section 10 — Prediction Charts (`visualizer.py`)

### 10.1 LSTM vs Random Forest — Validation Set


In [ ]:
def plot_predictions(train, valid, future_df=None):
    """
    Overlay training data, validation actuals, LSTM predictions,
    RF predictions (if present), and optional future forecast.
    """
    plt.figure(figsize=(16, 8))
    title = f"{PRIMARY_TICKER_NAME} Stock Price Prediction — LSTM vs Random Forest"
    if future_df is not None:
        title += f" (+ {NUM_FUTURE_DAYS}-Day Forecast)"
    plt.title(title, fontsize=15)
    plt.xlabel("Date", fontsize=14)
    plt.ylabel("Close Price USD ($)", fontsize=14)

    plt.plot(train["Close"],  color="blue",  label="Training Data (Actual)")
    plt.plot(valid["Close"],  color="green", label="Validation Data (Actual)")

    if "Predictions" in valid.columns:
        plt.plot(valid["Predictions"],
                 color="red", linewidth=1.8, label="LSTM Predictions")

    if "RF Predictions" in valid.columns:
        plt.plot(valid["RF Predictions"],
                 color="orange", linestyle="--", linewidth=1.8,
                 label="Random Forest Predictions")

    if future_df is not None:
        plt.plot(future_df["Future Predictions"],
                 color="purple", linestyle="--", linewidth=2,
                 label=f"Future {NUM_FUTURE_DAYS} Days (LSTM Forecast)")

    plt.legend(loc="lower right", fontsize=11)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_predictions(train_slice, valid_slice)

## 🔮 Section 11 — 60-Day Future Forecast (LSTM)

The LSTM autoregressively predicts the next 60 days by feeding each prediction  
back as input for the next step.


In [ ]:
def build_future_sequence(scaled_data, model, scaler):
    """Autoregressively forecast NUM_FUTURE_DAYS steps beyond the last known date."""
    current_sequence     = scaled_data[-LOOKBACK_WINDOW:].copy()
    future_preds_scaled  = []

    print(f"Generating {NUM_FUTURE_DAYS} future predictions…")
    for _ in range(NUM_FUTURE_DAYS):
        seq        = current_sequence.reshape(1, LOOKBACK_WINDOW, 1)
        scaled_pred = model.predict(seq, verbose=0)
        future_preds_scaled.append(scaled_pred[0][0])
        current_sequence = np.append(current_sequence[1:], scaled_pred, axis=0)

    future_predictions_usd = scaler.inverse_transform(
        np.array(future_preds_scaled).reshape(-1, 1)
    )
    last_date    = df.index[-1]
    future_dates = [last_date + timedelta(days=x) for x in range(1, NUM_FUTURE_DAYS + 1)]
    print("✅ Forecast complete.")
    return future_predictions_usd, future_dates

future_predictions_usd, future_dates = build_future_sequence(scaled_data, lstm, scaler)
future_df = pd.DataFrame(
    index=future_dates,
    data={"Future Predictions": future_predictions_usd.flatten()}
)
future_df.head(10)

### 11.2 Final Chart — Training + Validation + 60-Day Forecast

In [ ]:
train_full = df[:training_data_len]
valid_full = df[training_data_len:].copy()
valid_full["Predictions"]    = predictions_lstm[:len(valid_full)]
valid_full["RF Predictions"] = predictions_rf[:len(valid_full)]

plot_predictions(train_full, valid_full, future_df=future_df)

## 🌲 Section 12 — Random Forest Feature Importances

Which of the 60 lag days matters most to the forest?


In [ ]:
importances = rf.feature_importances
lag_labels  = [f"t-{60 - i}" for i in range(60)]

plt.figure(figsize=(16, 4))
plt.bar(lag_labels, importances, color="darkorange", edgecolor="white", width=0.8)
plt.xticks(lag_labels[::5], rotation=45, ha="right")
plt.title("Random Forest — Feature Importances (60 Lag Days)", fontsize=14)
plt.xlabel("Lag Day")
plt.ylabel("Importance Score")
plt.tight_layout()
plt.show()

most_important = lag_labels[np.argmax(importances)]
print(f"Most important lag: {most_important}  (importance = {importances.max():.4f})")

## ✅ Section 13 — Summary & Conclusions


In [ ]:
print("="*55)
print("         PROJECT RESULTS SUMMARY")
print("="*55)
print(f"  Ticker         : {PRIMARY_TICKER} ({PRIMARY_TICKER_NAME})")
print(f"  Training data  : {LSTM_START_DATE} → present")
print(f"  Lookback window: {LOOKBACK_WINDOW} trading days")
print(f"  Train/Test split: {int(TRAIN_SPLIT_RATIO*100)}% / {int((1-TRAIN_SPLIT_RATIO)*100)}%")
print()
print(f"  {'Model':<22} {'RMSE':>10}")
print("  " + "-"*33)
print(f"  {'LSTM':<22} {metrics_lstm['rmse']:>10.4f}")
print(f"  {'Random Forest':<22} {metrics_rf['rmse']:>10.4f}")
print()
better = "LSTM" if metrics_lstm['rmse'] < metrics_rf['rmse'] else "Random Forest"
print(f"  ✅ Better model (lower RMSE): {better}")
print(f"  🔮 {NUM_FUTURE_DAYS}-day LSTM forecast generated.")
print("="*55)

---

### Key Takeaways

| Aspect | LSTM | Random Forest |
|--------|------|---------------|
| **Type** | Deep Learning (Sequential) | Ensemble ML (Non-sequential) |
| **Input** | 3D `(samples, 60, 1)` | Flattened to 2D `(samples, 60)` |
| **Learns temporal order?** | ✅ Yes | ❌ No |
| **Training time** | Slower | Fast |
| **Extrapolation** | Better (trend-aware) | Limited (bounded by training range) |
| **Future forecasting** | Autoregressive rollout | Not easily extended |

### Future Work
- Add more tickers and a portfolio-level view
- Try **GRU** or **Transformer** models as additional baselines
- Add **sentiment analysis** from financial news as an extra feature
- Use **walk-forward validation** for a more realistic backtesting evaluation

---
*Project by — AI & Machine Learning Course*
